# PitWall Intelligence — Notebook 02: Preprocessing
**Sprint 1 | Data Acquisition & EDA**

Cleans raw data and builds the key analytical tables needed for EDA and the BVI model:
- `team_season_stats` — per-constructor per-season aggregates (points, wins, podiums, DNFs, grid positions)
- `qual_times_clean` — qualifying times converted to seconds
- Missing value audit
- Exports clean CSVs for EDA notebook

In [1]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/PitWall'
DB_PATH      = f'{PROJECT_ROOT}/data/pitwall.db'
EXPORTS_DIR  = f'{PROJECT_ROOT}/data/exports'

pd.set_option('display.max_columns', None)
print('Ready.')

Mounted at /content/drive
Ready.


In [2]:
# ── Cell 2: Load raw tables ─────────────────────────────────────────────────
with sqlite3.connect(DB_PATH) as conn:
    results  = pd.read_sql('SELECT * FROM results',  conn)
    races    = pd.read_sql('SELECT * FROM races',    conn)
    qual     = pd.read_sql('SELECT * FROM qualifying', conn)
    cs       = pd.read_sql('SELECT * FROM constructor_standings', conn)
    constrs  = pd.read_sql('SELECT * FROM constructors', conn)

print(f'results:   {results.shape}')
print(f'races:     {races.shape}')
print(f'qual:      {qual.shape}')
print(f'cs:        {cs.shape}')
results.head(3)

results:   (4626, 14)
races:     (228, 9)
qual:      (4610, 10)
cs:        (112, 7)


,result_id,race_id,season,round,driver_id,constructor_id,grid,position,position_text,position_order,points,laps,status,fastest_lap_rank
0,1,1,2014,1,rosberg,mercedes,3,1,1,0,25.0,57,Finished,1.0
1,2,1,2014,1,kevin_magnussen,mclaren,4,2,2,0,18.0,57,Finished,6.0
2,3,1,2014,1,button,mclaren,10,3,3,0,15.0,57,Finished,5.0


In [3]:
# ── Cell 3: Missing value audit ─────────────────────────────────────────────
def missing_audit(df, name):
    pct = (df.isnull().sum() / len(df) * 100).round(1)
    pct = pct[pct > 0].sort_values(ascending=False)
    if pct.empty:
        print(f'{name}: no missing values')
    else:
        print(f'\n{name} missing %:')
        print(pct.to_string())

for df, name in [(results,'results'), (qual,'qualifying'), (cs,'constructor_standings')]:
    missing_audit(df, name)


results missing %:
fastest_lap_rank    4.3

qualifying missing %:
q3    51.8
q2    26.4
constructor_standings: no missing values


In [4]:
# ── Cell 4: Clean results ───────────────────────────────────────────────────
res = results.copy()

# Classify finish status
DNF_KEYWORDS = ['Accident','Collision','Mechanical','Engine','Gearbox','Hydraulics',
                'Electrical','Suspension','Brakes','Wheel','Tyre','Puncture',
                'Power Unit','Overheating','Oil','Retired']

res['finished'] = res['position'].notna().astype(int)
res['is_dnf']   = res['status'].apply(
    lambda s: int(any(k.lower() in str(s).lower() for k in DNF_KEYWORDS))
)
res['on_podium']   = (res['position'].fillna(99) <= 3).astype(int)
res['points_flag'] = (res['points'] > 0).astype(int)

print(res[['status','is_dnf','on_podium','points']].sample(10))

        status  is_dnf  on_podium  points
1088   +2 Laps       0          0     0.0
2235    +1 Lap       0          0     2.0
839   Finished       0          0     0.0
3826   Retired       1          0     0.0
3190  Finished       0          0    13.0
2841    +1 Lap       0          0     0.0
2814  Finished       0          0     4.0
1067   Gearbox       1          0     0.0
4414  Finished       0          0     2.0
3656    +1 Lap       0          0     1.0


In [5]:
# ── Cell 5: Build team_season_stats ─────────────────────────────────────────
# For BVI: season-level aggregates per constructor

agg = res.groupby(['season','constructor_id']).agg(
    total_points   = ('points',       'sum'),
    total_wins     = ('position',     lambda x: (x == 1).sum()),
    total_podiums  = ('on_podium',    'sum'),
    total_dnfs     = ('is_dnf',       'sum'),
    races_entered  = ('race_id',      'nunique'),
    avg_grid       = ('grid',         'mean'),
    avg_finish_pos = ('position_order','mean'),
    finish_rate    = ('finished',     'mean'),
    points_scoring_rate = ('points_flag','mean')
).reset_index()

# Merge with final constructor standings for official season rank
final_cs = cs.sort_values('round').groupby(['season','constructor_id']).last().reset_index()
final_cs = final_cs[['season','constructor_id','position','wins']].rename(
    columns={'position':'season_rank','wins':'official_wins'})

team_season = agg.merge(final_cs, on=['season','constructor_id'], how='left')
team_season = team_season.merge(constrs[['constructor_id','name','nationality']],
                                on='constructor_id', how='left')

# Normalise within-season for BVI inputs
for col in ['total_points','total_wins','total_podiums']:
    season_max = team_season.groupby('season')[col].transform('max')
    team_season[f'{col}_norm'] = (team_season[col] / season_max.replace(0, np.nan)).fillna(0)

team_season = team_season.sort_values(['season','season_rank'])
team_season.head(10)

,season,constructor_id,total_points,total_wins,total_podiums,total_dnfs,races_entered,avg_grid,avg_finish_pos,finish_rate,points_scoring_rate,season_rank,official_wins,name,nationality,total_points_norm,total_wins_norm,total_podiums_norm
6,2014,mercedes,701.0,16,31,5,19,2.947368,0.0,1.0,0.842105,1,16,Mercedes,German,1.000000,1.0000,1.000000
7,2014,red_bull,405.0,3,12,3,19,7.026316,0.0,1.0,0.842105,2,3,Red Bull,Austrian,0.577746,0.1875,0.387097
10,2014,williams,320.0,0,9,5,19,6.842105,0.0,1.0,0.736842,3,0,Williams,British,0.456491,0.0000,0.290323
1,2014,ferrari,216.0,0,2,2,19,7.947368,0.0,1.0,0.789474,4,0,Ferrari,Italian,0.308131,0.0000,0.064516
5,2014,mclaren,181.0,0,2,1,19,8.631579,0.0,1.0,0.657895,5,0,McLaren,British,0.258203,0.0000,0.064516
2,2014,force_india,155.0,0,1,6,19,11.500000,0.0,1.0,0.710526,6,0,Force India,Indian,0.221113,0.0000,0.032258
9,2014,toro_rosso,30.0,0,0,8,19,11.447368,0.0,1.0,0.315789,7,0,Toro Rosso,Italian,0.042796,0.0000,0.000000
3,2014,lotus_f1,10.0,0,0,7,19,16.315789,0.0,1.0,0.078947,8,0,Lotus F1,British,0.014265,0.0000,0.000000
4,2014,marussia,2.0,0,0,7,16,18.677419,0.0,1.0,0.032258,9,0,Marussia,Russian,0.002853,0.0000,0.000000
8,2014,sauber,0.0,0,0,13,19,15.184211,0.0,1.0,0.000000,10,0,Sauber,Swiss,0.000000,0.0000,0.000000


In [6]:
# ── Cell 6: Parse qualifying times to seconds ───────────────────────────────

def lap_to_sec(lap_str):
    """Convert '1:23.456' to 83.456 seconds. Returns NaN if missing."""
    if not lap_str or pd.isna(lap_str):
        return np.nan
    try:
        parts = str(lap_str).split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        return float(parts[0])
    except:
        return np.nan

qual_clean = qual.copy()
for col in ['q1','q2','q3']:
    qual_clean[f'{col}_sec'] = qual_clean[col].apply(lap_to_sec)

# Best time achieved in session
qual_clean['best_qual_sec'] = qual_clean[['q1_sec','q2_sec','q3_sec']].min(axis=1)

# Grid advantage: delta to pole (positive = slower than pole)
pole = qual_clean[qual_clean['position'] == 1][['season','round','best_qual_sec']]\
       .rename(columns={'best_qual_sec':'pole_sec'})
qual_clean = qual_clean.merge(pole, on=['season','round'], how='left')
qual_clean['gap_to_pole_sec'] = qual_clean['best_qual_sec'] - qual_clean['pole_sec']

print(qual_clean[['season','round','driver_id','position','best_qual_sec','gap_to_pole_sec']].head(10))

   season  round        driver_id  position  best_qual_sec  gap_to_pole_sec
0    2014      1         hamilton         1         91.699            0.000
1    2014      1        ricciardo         2         90.775           -0.924
2    2014      1          rosberg         3         92.564            0.865
3    2014      1  kevin_magnussen         4         90.949           -0.750
4    2014      1           alonso         5         91.388           -0.311
5    2014      1           vergne         6         93.488            1.789
6    2014      1       hulkenberg         7         93.893            2.194
7    2014      1            kvyat         8         93.777            2.078
8    2014      1            massa         9         91.228           -0.471
9    2014      1           bottas        10         91.601           -0.098


In [7]:
# ── Cell 7: Constructor qualifying summary ──────────────────────────────────
# Average gap to pole per constructor per season (proxy for pace/exposure)

qual_team = qual_clean.groupby(['season','constructor_id']).agg(
    avg_gap_to_pole = ('gap_to_pole_sec', 'mean'),
    avg_qual_pos    = ('position',         'mean'),
    q3_appearances  = ('q3_sec',           lambda x: x.notna().sum())
).reset_index()

team_season = team_season.merge(qual_team, on=['season','constructor_id'], how='left')
print(team_season[['season','name','total_points','season_rank','avg_gap_to_pole','q3_appearances']].head(20))

    season         name  total_points  season_rank  avg_gap_to_pole  \
0     2014     Mercedes         701.0            1         0.184730   
1     2014     Red Bull         405.0            2         1.123778   
2     2014     Williams         320.0            3         1.637842   
3     2014      Ferrari         216.0            4         1.983789   
4     2014      McLaren         181.0            5         1.628342   
5     2014  Force India         155.0            6         2.171684   
6     2014   Toro Rosso          30.0            7         2.170368   
7     2014     Lotus F1          10.0            8         3.240265   
8     2014     Marussia           2.0            9         4.410871   
9     2014       Sauber           0.0           10         3.165297   
10    2014     Caterham           0.0           11         5.524303   
11    2015     Mercedes         703.0            1         0.211079   
12    2015      Ferrari         428.0            2         1.139421   
13    

In [8]:
# ── Cell 8: Save to SQLite and export ───────────────────────────────────────

with sqlite3.connect(DB_PATH) as conn:
    team_season.to_sql('team_season_stats', conn, if_exists='replace', index=False)
    qual_clean.to_sql('qual_times_clean',   conn, if_exists='replace', index=False)
    print('Saved to SQLite.')

team_season.to_csv(f'{EXPORTS_DIR}/team_season_stats.csv', index=False)
qual_clean.to_csv(f'{EXPORTS_DIR}/qual_times_clean.csv',   index=False)
print('Exported CSVs.')
team_season.describe()

Saved to SQLite.
Exported CSVs.


,season,total_points,total_wins,total_podiums,total_dnfs,races_entered,avg_grid,avg_finish_pos,finish_rate,points_scoring_rate,season_rank,official_wins,total_points_norm,total_wins_norm,total_podiums_norm,avg_gap_to_pole,avg_qual_pos,q3_appearances
count,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000,112.0,112.0,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000
mean,2018.928571,207.066964,2.035714,6.107143,6.008929,20.660714,10.484119,0.0,1.0,0.491148,5.598214,2.035714,0.304787,0.149751,0.219219,1.933443,10.663551,19.767857
std,3.195476,225.896717,4.561746,9.533589,3.256305,1.896196,4.423356,0.0,0.0,0.308679,2.951438,4.561746,0.332407,0.315497,0.342287,1.183548,4.734657,14.138302
min,2014.000000,0.000000,0.000000,0.000000,0.000000,16.000000,1.815789,0.0,1.0,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.184730,1.815789,0.000000
25%,2016.000000,30.000000,0.000000,0.000000,4.000000,19.000000,6.815789,0.0,1.0,0.188312,3.000000,0.000000,0.044382,0.000000,0.000000,1.082767,6.773810,7.000000
50%,2019.000000,108.500000,0.000000,1.000000,5.000000,21.000000,10.866071,0.0,1.0,0.500000,6.000000,0.000000,0.161909,0.000000,0.031754,1.916071,10.947368,18.000000
75%,2022.000000,364.250000,1.000000,9.000000,8.000000,22.000000,13.837500,0.0,1.0,0.790969,8.000000,1.000000,0.523571,0.080420,0.300000,2.532682,14.372907,32.250000
max,2024.000000,790.000000,21.000000,33.000000,15.000000,24.000000,19.441176,0.0,1.0,0.975000,11.000000,21.000000,1.000000,1.000000,1.000000,6.350429,20.058824,45.000000
